In [1]:
#chapter2.7
import torch
import torch.nn as nn

input_ids = torch.tensor([2, 3, 5, 1])
print(input_ids)
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim) #lookup table that takes 1*1 input and returns 1*3 input. has 6 different input options
print(embedding_layer.weight)
print(embedding_layer(torch.tensor([3])))

tensor([2, 3, 5, 1])
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [2]:
#exercise 2.1
!pip install tiktoken
import tiktoken

text = "akwirw ier"
tokenizer = tiktoken.get_encoding("gpt2")

integers = tokenizer.encode(text)
print(integers)

for i in integers:
    print(tokenizer.decode_single_token_bytes(i))

print(tokenizer.decode(integers))

#yes, it can reconstruct the original string

[461, 86, 343, 86, 220, 959]
b'ak'
b'w'
b'ir'
b'w'
b' '
b'ier'
akwirw ier


In [3]:
#chapter2.8
#importing dataset 
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [4]:
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):#this will be wrapped by a dataloader so no external function needed
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)#this block generates slices for next token prediction
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]#e.g if max_length is 5, this will make a slice of 5 tokens and a slice of 6 tokens. the next token prediction task will be carried out on the 5 and the 6 will be used to backprop
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
#explanation - given 100 tokens, max len 5 and stride 10
#i will step from 0 to 90 (bounded by 100-5)
#on first step
#input_chunk will be tokens 0 to 5
#target_chunk will be 1 to 6
#can iterate through these chunks for next token prediction
#visual intuition
# %%%%%
#  #####
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [5]:
from torch.utils.data import DataLoader
#le data loader
def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [6]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim) #this is creating an embedding layer

max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader) #make the dataloader iterable
inputs, targets = next(data_iter)#iterate it once
print("Token IDs:\n", inputs)#print the inputs and shape
print("\nInputs shape:\n", inputs.shape)
token_embeddings = token_embedding_layer(inputs) #embed
#print(token_embeddings.shape)
#print(token_embeddings)

context_length = max_length #maximum length of possible input
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)#of context length * output dim
pos_embeddings = pos_embedding_layer(torch.arange(context_length))#so each output dim of the embedding table is changed with respect to position
print(pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings#add together
#print(input_embeddings.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])
torch.Size([4, 256])


In [7]:
#also listing 2.3
import re

class SimpleTokenizerV1:
	def __init__(self, vocab):
		self.str_to_int = vocab #A
		self.int_to_str = {i:s for s,i in vocab.items()} #B
	def encode(self, text): #C
		preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
		preprocessed = [item.strip() for item in preprocessed if item.strip()]
		ids = [self.str_to_int[s] for s in preprocessed]
		return ids
	def decode(self, ids): #D
		text = " ".join([self.int_to_str[i] for i in ids])
		text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) #E
		return text

In [8]:
#listing 2.3
import re

with open("the-verdict.txt", "r", encoding="utf-8") as f:
	raw_text = f.read()
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_words = sorted(list(set(preprocessed)))
vocab_size = len(all_words)
print(vocab_size)

vocab = {token:integer for integer,token in enumerate(all_words)}

tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with"""
ids = tokenizer.encode(text)
print(ids)
text = tokenizer.decode(ids)
print(text)

1159
[1, 58, 2, 872, 1013, 615, 541, 763, 5, 1155, 608, 5, 1, 69, 7, 39, 873, 1136]
" It' s the last he painted, you know," Mrs. Gisburn said with


In [9]:
#chapter 3.1

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)

query = inputs[1] #the embedding of the word 'journey'
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs): #provides each row of inputs and its index
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

#attn_scores_2 is the dot product between the query [inputs [1]] and all other vectors in inputs


attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

#attn_weights_2 is the softmax of the dot product between inputs[1] and all other rows in inputs

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
    #this is the dot product of attn_weights with every other entry
print(context_vec_2) #why is this 1x3?
                    #because it was embedded into three dimension vectors

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)
tensor([0.4419, 0.6515, 0.5683])


In [10]:
#chapter 3.3. 3.2 didn't have anything i made notes on
import torch

#INPUTS IS TOKENS AFTER EMBEDDING
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)

#this is a way of generating attention scores for all
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)#the crux of this is that it calculates the similarity between two vectors as a single number
print(attn_scores)
#the very core of the whole transformer thing
#calculate dot products between vectors
#put through neural network
#eventually have a good representation of how tokens relate to each other
#can use this as a basis for next token prediction

#much faster way of generating attention scores
attn_scores = inputs @ inputs.T #@ is used for matrix multiplication
print(attn_scores)

#just softmaxxing to normalise and get weights
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

#matrix multiplication between attn_weights and inputs
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1

In [11]:
#listing 3.1 
import torch.nn as nn
import torch

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T  # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) #nn.Linear objects take a matrix of d_in and output a matrix of d_out through a learnable transformation. that transformation can have a bias value. if qkv_bias is True, it has one
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x): #x is the embedded matrix
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [12]:
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)

x_2 = inputs[1] #taking first token
d_in = inputs.shape[1]#3 tokens
d_out = 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) #requires_grad not needed bc only doing inference
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) #requires_grad is used if backprop is used
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) #3,2 casts 3d vectors to 2d vectors. choice of d_out is entirely random here

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print("printing query 2 to inspect it", query_2)

keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)#calculating logit value of token 2's attention to itself 
print(attn_score_22)

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

printing query 2 to inspect it tensor([0.4306, 1.4551])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
tensor(1.8524)
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [13]:
#listing 3.3
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec




In [14]:
#chapter 3.5

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)

torch.manual_seed(789)
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2


sa_v2 = SelfAttention_v2(d_in, d_out)


queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)#same as before
print(attn_weights)

#as this llm is intended for generation, all tokens that are after the current token can be masked out. this is so the model learns to predict the next token as this data is not provided.
#if the model can look at tokens ahead of the current position, it becomes a classification model and not a next token prediction model

context_length = attn_scores.shape[0]
#mask_simple = torch.tril(torch.ones(context_length, context_length))
#print(mask_simple)

#masked_simple = attn_weights*mask_simple
#print(masked_simple)

#however, this is a bit computationally inefficient.

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)#torch.triu returns a triangle of a tensor. in this case it is a tensor of ones of contextlength^2. this is for causal masking
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)
#this is much fewer operations but still needs to be softmaxxed
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

#dropout is useful because it forces the model to 'think outside the box' and learn the relationships between tokens as opposed to memorising
#it is disabled during actual use

torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example)) #this puts some values to 0

torch.manual_seed(123)
print(dropout(attn_weights))

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)
tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000

In [15]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_length, dropout, qkv_bias
            )
            for _ in range(num_heads)] #_ is a throwaway variable and has no use. it just ensures that casual attention is created in the list
        )
    def forward(self, x):
            return torch.cat([head(x) for head in self.heads], dim=-1)

In [16]:
#exercise 3.1
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)
d_in = inputs.shape[1]
d_out = 2




torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T) #this is just oop in python
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)
print(sa_v1(inputs))

#see modified listing code for soln

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [17]:
#listing 3.5
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x) #all as normal

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        #here, the tensors are rearranged to have the following form [b, num_tokens, num_heads, head_dim]
        #this means that all the head operations can be done simultaneously as opposed to sequentially
        #as it's now just the dot product of two matrices being found


        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)


        attn_scores = queries @ keys.transpose(2, 3) #swaps num_heads and head_dim. then dot products to get attention scores as normal
        #attention score is how relevant tokens are to each other
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        #normalising and dropping out attention scores

        context_vec = (attn_weights @ values).transpose(1, 2)
        #context_vec is weighted sum of all vectors for each token

        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )

        context_vec = self.out_proj(context_vec)
        #out_proj is needed because without it, the output from the heads is just stacked. out_proj is needed in order
        return context_vec

In [18]:
#chapter 3.6
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2 #change d_out to 1 for exercise 3.2

mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573], # 1 batch, 2 heads [each block is a head], 3 rows in each block per token, 4 token dimensions
[0.8993, 0.0390, 0.9268, 0.7388],
[0.7179, 0.7058, 0.9156, 0.4340]],
[[0.0772, 0.3565, 0.1479, 0.5331],
[0.4066, 0.2318, 0.4545, 0.9737],
[0.4606, 0.5159, 0.4220, 0.5786]]]])
print("a.transpose below")
print(a.transpose(2, 3))
#swaps num_tokens and head_dim

print(a @ a.transpose(2, 3)) #the transpose is necessary so that the dot product can be found. kinda like 'finding the square' so a map of how tokens relate to each other can be made


torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
a.transpose below
tensor([[[[0.2745, 0.8993, 0.7179],
          [0.6584, 0.0390, 0.7058],
          [0.2775, 0.9268, 0.9156],
          [0.8573, 0.7388, 0.4340]],

         [[0.0772, 0.4066, 0.4606],
          [0.3565, 0.2318, 0.5159],
          [0.1479, 0.4545, 0.4220],
          [0.5331, 0.9737, 0.5786]]]])
tensor([[[[1.3208, 1.1631, 1.2879

In [19]:
#beneath here is chapter4. ctrl + f 4.1 if you want to see the chapter, keep reading if you want to see the base implementations 

In [20]:
GPT_CONFIG_124M = { #this defines the config for the modern
"vocab_size": 50257,
"context_length": 1024,
"emb_dim": 768,
"n_heads": 12,
"n_layers": 12,
"drop_rate": 0.1,
"qkv_bias": False
}

In [21]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x): #this is an approximation of the the GELU function
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3)))
            )


In [22]:

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
        GELU(), #this is a basic inverse hourglass nn
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
    ) ##nn.Linear carries out a linear transform on the input matrix (w = Ax + B)>

    def forward(self, x):
        return self.layers(x)


In [23]:

class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut

        # Define the layers as a list of Sequential modules
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), nn.GELU())
        ])

    def forward(self, x):
        # Pass input through each layer, optionally adding shortcut connections
        for layer in self.layers:
            layer_output = layer(x)
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
                #shortcut connections sum the output of the network back into the residual stream
                #as is shown in the maths above
        return x



In [24]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5 #added to prevent division by 0 during normalisation
        self.scale = nn.Parameter(torch.ones(emb_dim)) #variance
        self.shift = nn.Parameter(torch.zeros(emb_dim)) #mean
    def forward(self, x):                              #scale and shift allow the model to reintroduce variability back in if needed. they are learnable parameters so can be altered if the gradient explodes
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [25]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x) #this sets (100*drop_rate)% of neurons to 0 and scales up the output of the other neurons to compensate
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x


In [26]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        ) #this takes emb_dim sized vectors from the output of the transformer blocks and then casts it to a vocab_size vector. this is a map of the most likely next token (logits)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [27]:
#4.1
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)
#batch is just equivalent to the tokenised verions of txt

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output shape:", logits.shape)
#logits is just the outputs of txt
print(logits)

#4.2
torch.manual_seed(123)
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())#linear transformation + relu
out = layer(batch_example)
print(out)

mean = out.mean(dim=-1, keepdim=True)#dim = -1 -> calculate across last dimensions. dim = 0 -> calculate across first dimension etc. dim = 1 is columns in the case of a 2d vector
var = out.var(dim=-1, keepdim=True)#keepdim = True -> maintain input dimensions
print("Mean:\n", mean)
print("Variance:\n", var)


out_norm = (out - mean) / torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Normalized layer outputs:\n", out_norm)
print("Mean:\n", mean)
print("Variance:\n", var)

#layernorm normalises each layer to have a mean value of 0 and standard deviation of 1
#this is important because of the vanishing/exploding gradient problem
#computers can make mistakes with very small or large numbers
#it is possible to reach these if the layers have very large or small gradients

ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

#4.3

ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 3, 768)
out = ffn(x)
print(out.shape)

layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123)

model_without_shortcut = ExampleDeepNeuralNetwork(
layer_sizes, use_shortcut=False
)


def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])
    loss = nn.MSELoss() #mean squared error loss
    loss = loss(output, target) #target is all 0s
    loss.backward()
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

print_gradients(model_without_shortcut, sample_input)
#this shows the vanishing gradients as small gradients are multiplied by even smaller gradients

print("the model below uses shortcuts")

torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)

print_gradients(model_with_shortcut, sample_input)

torch.manual_seed(123)
x = torch.rand(2, 4, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

print("printing shapes of full gpt model input/output \n")
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

print("printing params of full gpt model \n")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}") #there is a discrepancy between the number of params declared in config (124 M) and printed here (163 M)
#this is because there are different layers for casting in and out of token_ids. using the same layer is called 'weight tying'. using it can decrease performance but does decrease memory performance
#as further insight, out head is equal to (vocab_size * embedding_dimensions)

total_size_bytes = total_params * 4
total_size_mb = total_size_bytes / (1024 * 1024)
print(f"Total size of the model: {total_size_mb:.2f} MB")

start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

model.eval()


def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:] #crops current context to maximum size
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :] #this is because the model outputs predictions based on each token. example - given tokens 1, 2, 3, 4, 5, the model will output predictions for 2, 3, 4, 5, 6. only the prediction for token 6 is needed
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx



out = generate_text_simple(
    model=model,
idx=encoded_tensor,
max_new_tokens=6,
context_size=GPT_CONFIG_124M["context_length"]
)

print("output:", out)
print("Output length:", len(out[0]))

decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.1381,  0.0077, -0.1963,  ..., -0.0222, -0.1060,  0.1717],
         [ 0.3865, -0.8408, -0.6564,  ..., -0.5163,  0.2369, -0.3357],
         [ 0.6989, -0.1829, -0.1631,  ...,  0.1472, -0.6504, -0.0056],
         [-0.4290,  0.1669, -0.1258,  ...,  1.1579,  0.5303, -0.5549]],

        [[ 0.1094, -0.2894, -0.1467,  ..., -0.0557,  0.2911, -0.2824],
         [ 0.0882, -0.3552, -0.3527,  ...,  1.2930,  0.0053,  0.1898],
         [ 0.6091,  0.4702, -0.4094,  ...,  0.7688,  0.3787, -0.1974],
         [-0.0612, -0.0737,  0.4751,  ...,  1.2463, -0.3834,  0.0609]]],
       grad_fn=<UnsafeViewBackward0>)
tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)
Mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBac

In [28]:
#the output from the model above is gibberish. this because it has not been trained. below, i shall train the model to accurately reproduce text from the verdict

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())


def calc_loss_batch(input_batch, target_batch, model, device):  # wrapper function for loss calculation
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()   # check the chances of the correct tokens being picked by looking at the logits. see above for implementation
    )  # check the chances of the correct tokens being picked by looking at the logits. see above for implementation
    # syntax -> pass flattened logits and targets
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):  # wraps calc_loss_batch and iterates over loader
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")  # error handling

    elif num_batches is None:
        num_batches = len(data_loader)  # data_loader

    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):  # iterates over batches in data_loader. data_loader is a construct that wraps dataset
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )  # calculate loss per batch
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches  # provide average loss per batch


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()  # sets internal training flag to false (i.e - turns dropout off)
    with torch.no_grad():  # turns off gradient tracking so no log of activations is kept. this is faster
        train_loss = calc_loss_loader(  # see above
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)  # start_context is tokenised and assigned to encoded
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded, max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()


def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):

    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()  # puts model into training mode by enabling dropout
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()  # resets optimiser from last run
            loss = calc_loss_batch(input_batch, target_batch, model, device)  # see comments above
            loss.backward()  # work out backprop of loss to see what caused the loss
            optimizer.step()  # and do something about the loss
            tokens_seen += input_batch.numel()  # .numel -> number of elements in input batch
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch + 1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

        generate_and_print_sample(
            model, tokenizer, device, start_context
        )  # generate sample text

    return train_losses, val_losses, track_tokens_seen


# torch.nn.cross_entropy (calculates cross entropy) -> calculate_loss_batch (calculates logits for batch and feeds into cross_ent)
def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator

    fig, ax1 = plt.subplots(figsize=(5, 3))
    ax1.plot(epochs_seen, train_losses, label="training loss")
    ax1.plot(
        epochs_seen, val_losses, linestyle="-.", label="validation loss"
    )
    ax1.set_xlabel("epochs")
    ax1.set_ylabel("loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.show()
    # epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
    # plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
    # when training loss drops beneath validation loss, the model is overfitting
    # this can be intuitively understood as the feeling when questions you don't expect appear on the test


def print_sampled_tokens(probas, inverse_vocab):
    sample = [torch.multinomial(probas, num_samples=1).item()
        for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")


def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature  
    
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
model.eval()

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

inputs = torch.tensor([[16833, 3626, 6100], # ["every effort moves"]
[40, 1107, 588]]) # ["I really like"]

targets = torch.tensor([[3626, 6100, 345 ],
[1107, 588, 11311]])
# [" effort moves you",
# " really like chocolate"]

with torch.no_grad():
    logits = model(inputs) #this produces 2 rows (batch size), 3 tokens, 50257 possible options. this shows an unadjusted row of floats
probas = torch.softmax(logits, dim=-1) #casting to probabilities which sum to 1

token_ids = torch.argmax(probas, dim=-1, keepdim=True) #choosing most likely tokens
print("Token IDs:\n", token_ids)

print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"
f" {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)
#for a given text, at each position (0,1,2), extract the probability the model assigned to the correct token at that position


avg_log_probas = torch.log(torch.cat((target_probas_1, target_probas_2))) #then concatenate these probabilities and take the log of them
print(avg_log_probas)

neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)

logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print("loss:", loss)

perplexity = torch.exp(loss)
print("perplexity:", perplexity)

file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("Characters:", total_characters)
print("Tokens:", total_tokens)# when temp is v low, behaviour approaches that of argmax


def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):  # iterate max_new_tokens times
        idx_cond = idx[:, -context_size:]  # this is the context length cut - from the start to -context_length, take all values
        with torch.no_grad():  # if ^^ was not included, the model would be given something longer than context length.
            logits = model (idx_cond)
        logits = logits[:, -1, :]  # model takes sequences of tokens sequentially from batch. model outputs predictions of the next token for all tokens in sequence. example for 1, 2, 3, 4 -> model will output predictions for tokens in 2, 3, 4, 5

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
            )
        if temperature > 0.0:
            logits = logits/temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


def main():
    torch.manual_seed(123)

    model = GPTModel(GPT_CONFIG_124M)
    model.eval()

    start_context = "Every effort moves you"
    tokenizer = tiktoken.get_encoding("gpt2")

    token_ids = generate_text_simple(
        model=model,
        idx=text_to_token_ids(start_context, tokenizer),
        max_new_tokens=10,
        context_size=GPT_CONFIG_124M["context_length"]
    )

    print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

    inputs = torch.tensor([[16833, 3626, 6100], [40, 1107, 588]])
    # ["every effort moves", "I really like"] #both of these are shifted by one token

    targets = torch.tensor([[3626, 6100, 345 ], [1107, 588, 11311]])
    # [" effort moves you", " really like chocolate"]

    with torch.no_grad():              #disables gradient tracking
        logits = model(inputs)     #logits are floats essentially
    probas = torch.softmax(logits, dim=-1) #this gets cast to probabilities which sum to 1
    print(probas.shape) #this is a 2,3,50257
    #i.e, 2 inputs of 3 words, with 50257 predictions about what could follow each

    token_ids = torch.argmax(probas, dim=-1, keepdim=True) #choose the highest values of probas from the last dimension and preserve the other dimensions of the 'slice' with the highest value. for example, see below
    print("Token IDs:\n", token_ids) #this has a trailing dimension. despite being a (2,2), it has 3 brackets. the highest values of the 50257 have been taken for each other dimension. this collapses the matrix to a 2,3,1.

    print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
    print("Outputs batch 1:"
    f" {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

    text_idx = 0
    target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]] #from probas, select from the first batch, all(in this case, all means 3) of the tokens, and then the values corresponding to the correct answers.
    print("Text 1:", target_probas_1)
    text_idx = 1
    target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]] #same here
    print("Text 2:", target_probas_2)
    #each probas contains the probabilities of the correct answer being picked

    log_probas = torch.log(torch.cat((target_probas_1, target_probas_2))) #make a tensor of length 6 by concating the correct possibilities of the 2 strings in the batch and take the log. the log is used because it converts what would be multiplication of very small/large numbers into addition of moderately sized numbers
    print(log_probas)

    avg_log_probas = torch.mean(log_probas) #take the mean of the logs
    print(avg_log_probas)

    #note to self - this is an implementation of cross-entropy. it works out the probability of the correct token being picked over a batch by selecting the token ids of the correct tokens from logits.

    neg_avg_log_probas = avg_log_probas * -1 #make it a positive number (the aim will be to make it smaller)
    print(neg_avg_log_probas)

    print("Logits shape:", logits.shape)
    print("Targets shape:", targets.shape)

    logits_flat = logits.flatten(0, 1)
    targets_flat = targets.flatten()
    print("Flattened logits:", logits_flat.shape)
    print("Flattened targets:", targets_flat.shape)

    file_path = "the-verdict.txt"
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

    total_characters = len(text_data)
    total_tokens = len(tokenizer.encode(text_data))
    print("Characters:", total_characters)
    print("Tokens:", total_tokens)
    
    train_ratio = 0.90
    split_idx = int(train_ratio * len(text_data))
    train_data = text_data[:split_idx]
    eval_data = text_data[split_idx:]

    train_loader = create_dataloader(
        train_data,
        batch_size=2,
        max_length=GPT_CONFIG_124M["context_length"],
        stride=GPT_CONFIG_124M["context_length"],
        drop_last=True,
        shuffle=True,
        num_workers=0
    )

    val_loader = create_dataloader(
        eval_data,
        batch_size=2,
        max_length=GPT_CONFIG_124M["context_length"],
        stride=GPT_CONFIG_124M["context_length"],
        drop_last=False,
        shuffle=False,
        num_workers=0
    )

    #print("Train loader:")
    #for x, y in train_loader:
    #    print(x.shape, y.shape) #9 2x256

    #print("\nValidation loader:")
    #for x, y in val_loader:
    #    print(x.shape, y.shape) #1 2x256
    """
       #i ran this all on my pc and got weird cuda memory issues. commented out bc of migration to aws


       model = GPTModel(GPT_CONFIG_124M)
       model.to(device)
       device_under_test = device
        free, total = torch.cuda.mem_get_info(device_under_test)
       mem_used_MB = (total - free) / 1024 ** 2
       print("mem used before train", mem_used_MB)

    device_under_test = device
    free, total = torch.cuda.mem_get_info(device_under_test)
    mem_used_MB = (total - free) / 1024 ** 2
    print("mem used after optimiser created", mem_used_MB)
    """
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

    num_epochs = 10 #go over dataset 10x

    train_losses, val_losses, tokens_seen = train_model_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=num_epochs, eval_freq=5, eval_iter=5,
        start_context="Every effort moves you", tokenizer=tokenizer
    )

    #device_under_test = torch.device('cuda:0')
    #free, total = torch.cuda.mem_get_info(device_under_test)
    #mem_used_MB = (total - free) / 1024 ** 2
    #print("mem used after train", mem_used_MB)

    model.to("cpu")
    model.eval()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = tiktoken.get_encoding("gpt2")
    token_ids = generate_text_simple(
        model=model,
        idx=text_to_token_ids("every effort moves you ", tokenizer),
        max_new_tokens=25,
        context_size=GPT_CONFIG_124M["context_length"]
    )
    print("output text:\n", token_ids_to_text(token_ids, tokenizer))

    vocab = {
    "closer": 0,
    "every": 1,
    "effort": 2,
    "forward": 3,
    "inches": 4,
    "moves": 5,
    "pizza": 6,
    "toward": 7,
    "you": 8,
    }
    inverse_vocab = {v: k for k, v in vocab.items()} #this loops over vocab (for k,v in vocab.items) and writes it backwards into a new dictionary (v:k)

    next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
    )

    probas = torch.softmax(next_token_logits, dim=0) #softmax to get probabilities
    next_token_id = torch.argmax(probas).item() #choose most likely
    print(inverse_vocab[next_token_id])

    next_token_id = torch.multinomial(probas, num_samples=1).item() #samples according to distribution
    print(inverse_vocab[next_token_id])

    print_sampled_tokens(probas, inverse_vocab)

    # exercise 5.1 - chance of pizza being returned below
    # pizza is at position 6 so just read out values

    temp_altered = next_token_logits/5
    probs = torch.softmax(temp_altered, dim=0)
    print("exercise 5.1 ans:", probs[6]*100)

    top_k = 3
    top_logits, top_pos = torch.topk(next_token_logits, top_k) #returns keys and values of 3 highest  from each dimension
    print("Top logits:", top_logits)
    print("Top positions:", top_pos)

    new_logits = torch.where(
        condition=next_token_logits < top_logits[-1], #create a boolean mask for all values that are top_logits[-1] (-1 takes the last value and top_k automatically sorts by size)
        input=torch.tensor(float('-inf')), #cover with -inf tensors if true
        other=next_token_logits #carry out on next_token_logits
    )
    print(new_logits)

    topk_probas = torch.softmax(new_logits, dim=0)
    print(topk_probas)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids("every step moves you forward", tokenizer),
        max_new_tokens=70,
        context_size=GPT_CONFIG_124M["context_length"],
        top_k=25,
        temperature=1.4
    )
    print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

    #exercise 5.2 - high temp is better for situations where repeatability is bad - for example, roleplaying
    #exercise 5.3 - low temp is closest to argmax

    #device = torch.device('cuda:0')
    #free, total = torch.cuda.mem_get_info(device)
    #mem_used_MB = (total - free) / 1024 ** 2
    #print("mem used before save, wipe and load,", mem_used_MB)

    torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
    )

    print("model successfully saved")

    #import gc
    #del model, optimizer
    #gc.collect()
    #torch.cuda.synchronize()
    #torch.cuda.empty_cache()

    #device = torch.device('cuda:0')
    #free, total = torch.cuda.mem_get_info(device)
    #mem_used_MB = (total - free) / 1024 ** 2
    #print("mem used after save and wipe", mem_used_MB)



if __name__ == "__main__":
    main()










Output text:
 Every effort moves you confusionbladeMs synthesis assembleatefulumerous Technology Accountability 82
Token IDs:
 tensor([[[32036],
         [48510],
         [18153]],

        [[21505],
         [ 3506],
         [46416]]])
Targets batch 1:  effort moves you
Outputs batch 1:  LevyDKSports
Text 1: tensor([1.8517e-05, 1.5020e-05, 7.2692e-06])
Text 2: tensor([2.8445e-05, 2.9457e-05, 2.5030e-05])
tensor([-10.8968, -11.1061, -11.8319, -10.4676, -10.4326, -10.5954])
tensor([10.8968, 11.1061, 11.8319, 10.4676, 10.4326, 10.5954])
Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])
Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])
loss: tensor(10.8884)
perplexity: tensor(53551.6562)
Characters: 20479
Tokens: 5145
Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren
torch.Size([2, 3, 50257])
Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [4

In [ ]:
torch.manual_seed(123)
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
model.eval()

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

inputs = torch.tensor([[16833, 3626, 6100], # ["every effort moves"]
[40, 1107, 588]]) # ["I really like"]

targets = torch.tensor([[3626, 6100, 345 ],
[1107, 588, 11311]])
# [" effort moves you",
# " really like chocolate"]

with torch.no_grad():
    logits = model(inputs) #this produces 2 rows (batch size), 3 tokens, 50257 possible options. this shows an unadjusted row of floats
probas = torch.softmax(logits, dim=-1) #casting to probabilities which sum to 1

token_ids = torch.argmax(probas, dim=-1, keepdim=True) #choosing most likely tokens
print("Token IDs:\n", token_ids)

print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"
f" {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)
#for a given text, at each position (0,1,2), extract the probability the model assigned to the correct token at that position


avg_log_probas = torch.log(torch.cat((target_probas_1, target_probas_2))) #then concatenate these probabilities and take the log of them
print(avg_log_probas)

neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)

logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print("loss:", loss)

perplexity = torch.exp(loss)
print("perplexity:", perplexity)

file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("Characters:", total_characters)
print("Tokens:", total_tokens)